In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [24]:
df_original = pd.read_csv("logistics_data.csv")

df_original.head()

,Month_Name,Month_Number,Hours,Time,Date,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,...,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,Mar,3,0,00:11:14,20.03.24,20.03.24 00:11,Truck_7,-65.74,11.25,390,...,27.0,67.8,Detour,38,320,4,NaN,60.1,285,1
1,Oct,10,7,07:53:51,30.10.24,30.10.24 07:53,Truck_6,22.27,-131.71,491,...,22.5,54.3,Heavy,16,439,7,Weather,80.9,174,1
2,Jul,7,18,18:42:48,29.07.24,29.07.24 18:42,Truck_10,54.92,79.55,190,...,25.2,62.2,Detour,34,355,3,NaN,99.2,260,0
3,Oct,10,0,00:50:54,28.10.24,28.10.24 00:50,Truck_9,42.39,-1.48,330,...,25.4,52.3,Heavy,37,227,5,Traffic,97.4,160,1
4,Sep,9,15,15:52:58,27.09.24,27.09.24 15:52,Truck_7,-65.85,47.95,480,...,20.5,57.2,Clear,56,197,6,NaN,71.6,270,1


In [25]:
df_model = df_original.copy()

In [27]:
df_model = df_model.drop(columns=[
    'Shipment_Status',
    'Logistics_Delay_Reason',
    'Month_Name',
    'Timestamp',
    'Date',
    'Time'
], errors='ignore')

In [28]:
df_model = pd.get_dummies(
    df_model,
    columns=['Traffic_Status'],
    drop_first=True
)

In [29]:
df_model = df_model.drop(columns=['Asset_ID'], errors='ignore')

In [31]:
df_model = df_model.fillna(0)

In [32]:
X = df_model.drop('Logistics_Delay', axis=1)
y = df_model['Logistics_Delay']

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
df_original['Delay_Probability'] = model.predict_proba(X)[:,1]

In [37]:
def risk(p):
    if p > 0.7:
        return "High Risk"
    elif p > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df_original['risk_level'] = df_original['Delay_Probability'].apply(risk)

In [38]:
df_original.columns

Index(['Month_Name', 'Month_Number', 'Hours', 'Time', 'Date', 'Timestamp',
       'Asset_ID', 'Latitude', 'Longitude', 'Inventory_Level',
       'Shipment_Status', 'Temperature', 'Humidity', 'Traffic_Status',
       'Waiting_Time', 'User_Transaction_Amount', 'User_Purchase_Frequency',
       'Logistics_Delay_Reason', 'Asset_Utilization', 'Demand_Forecast',
       'Logistics_Delay', 'Delay_Probability', 'risk_level'],
      dtype='object')

In [39]:
df_original.to_csv("logistics_with_predictions.csv", index=False)